# 01 — RF Baseline LOLO
# Giai đoạn 2 — Mục 2.1 — Random Forest baseline với LOLO
**Đầu ra**: `outputs/tables/rf_baseline_lolo_results.csv`

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from common import training

In [3]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
feature_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/features_mlp.parquet")
feature_cols = [col for col in feature_df.columns if col.startswith(('time_', 'order_', 'envelope_'))]
print(f"Số đặc trưng: {len(feature_cols)}")

Số đặc trưng: 32


In [8]:
def rf_factory():
    return RandomForestClassifier(n_estimators=100, random_state=42)

per_fold, summary = training.run_lolo_evaluation(
    feature_df,
    feature_cols=feature_cols,
    estimator_factory=rf_factory,
    label_col='label',
    load_col='load_hp',
    val_ratio=0.2,
    seed=42,
    loads=(0, 1, 2, 3),
    use_val_for_fit=True  # Train trên train+val
)

In [9]:
per_fold.to_csv(TABLES_DIR / "rf_baseline_lolo_per_fold.csv", index=False)
summary_df = pd.DataFrame([summary])
summary_df.to_csv(TABLES_DIR / "rf_baseline_lolo_summary.csv", index=False)

print("\n=== Kết quả RF Baseline ===")
print(f"Accuracy: {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print(f"F1-macro: {summary['f1_macro_mean']:.4f} ± {summary['f1_macro_std']:.4f}")
per_fold


=== Kết quả RF Baseline ===
Accuracy: 1.0000 ± 0.0000
F1-macro: 1.0000 ± 0.0000


,fold_name,test_load,trainval_loads,n_train,n_val,n_test,accuracy,f1_macro
0,test_load_0,0,"[1, 2, 3]",24,6,10,1.0,1.0
1,test_load_1,1,"[0, 2, 3]",24,6,10,1.0,1.0
2,test_load_2,2,"[0, 1, 3]",24,6,10,1.0,1.0
3,test_load_3,3,"[0, 1, 2]",24,6,10,1.0,1.0
